# Citi Bike Station Health & Rebalancing Analytics

**Author:** Serena Zhang  
**Data:** August 2026 Citi Bike trips  
**Goal:** Build a reliable station-hour prioritization framework for bike- and dock-availability pressure.

This notebook mirrors the project workflow used for the portfolio dashboard:

1. Load six Citi Bike CSV files with DuckDB
2. Validate trip counts and schema
3. Audit trip-duration behavior
4. Normalize inconsistent station identifiers
5. Build arrivals and departures
6. Create a complete station × date × hour grid
7. Calculate imbalance, activity, and persistence KPIs
8. Build a composite Priority Score
9. Export a Tableau-ready station-hour dataset

> **Important:** The analysis identifies *availability pressure signals*. It does not claim that stations were definitively empty or full because real-time bike inventory and dock-capacity data are not included.

## 1. Setup

Install/import the required packages.

If you run this notebook in Google Colab, update `DATA_GLOB` to the folder containing the six August 2026 Citi Bike CSV files in your Google Drive.

In [ ]:
# Uncomment in a fresh Colab environment if needed:
# !pip install duckdb pandas pyarrow -q

import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### Optional: mount Google Drive in Colab

In [ ]:
# Run only in Google Colab if your files are stored in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Data path

Replace the placeholder below with the location of your extracted August 2026 Citi Bike CSV files.

Example for Colab:

`/content/drive/MyDrive/CitiBike_August_2026/*.csv`

In [ ]:
DATA_GLOB = "/path/to/202608-citibike-tripdata/*.csv"

con = duckdb.connect()

## 3. Load the six CSV files with DuckDB

In [ ]:
con.execute(f'''
CREATE OR REPLACE VIEW raw_trips AS
SELECT *
FROM read_csv_auto(
    '{DATA_GLOB}',
    union_by_name = true,
    header = true
)
''')

trip_count = con.execute("SELECT COUNT(*) FROM raw_trips").fetchone()[0]
print(f"Trips loaded: {trip_count:,}")

**Project validation target:** `5,246,236` trips.

Matching this total confirms that all six source files were loaded and helps prevent downstream analysis from being built on an incomplete extract.

In [ ]:
# Fail loudly if the project dataset is incomplete.
EXPECTED_TRIPS = 5_246_236

if trip_count != EXPECTED_TRIPS:
    print(f"Warning: expected {EXPECTED_TRIPS:,} trips, found {trip_count:,}.")
else:
    print("Trip-count QA passed.")

## 4. Understand the schema and grain

The raw dataset grain is:

> **One row = one bike trip**

Key fields include timestamps, start/end station identifiers and names, coordinates, ride type, and member/casual rider classification.

In [ ]:
schema = con.execute("DESCRIBE raw_trips").df()
schema

## 5. Trip-duration quality audit

Outliers are reviewed before any removal decision.

The project observed:

- Median duration: **9.78 minutes**
- 99th percentile: **65.83 minutes**
- Trips longer than 4 hours: **4,286**

The key principle is:

> **Unusual does not automatically mean invalid.**

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW trips_with_duration AS
SELECT
    *,
    date_diff(
        'second',
        CAST(started_at AS TIMESTAMP),
        CAST(ended_at AS TIMESTAMP)
    ) / 60.0 AS duration_min
FROM raw_trips
''')

duration_qa = con.execute('''
SELECT
    median(duration_min) AS median_duration_min,
    quantile_cont(duration_min, 0.99) AS p99_duration_min,
    SUM(CASE WHEN duration_min > 240 THEN 1 ELSE 0 END) AS trips_over_4h
FROM trips_with_duration
WHERE duration_min IS NOT NULL
''').df()

duration_qa

## 6. Canonical station-key normalization

Some physical stations appeared under multiple string representations.

Examples found during QA included:

- `5343.1` and `5343.10`
- `5303.06_`
- legitimate alphanumeric IDs such as `SYS016`

A naive numeric cast would destroy legitimate alphanumeric station IDs, so normalization must preserve them.

In [ ]:
con.execute(r'''
CREATE OR REPLACE MACRO canonical_station_id(x) AS (
    CASE
        WHEN x IS NULL OR trim(CAST(x AS VARCHAR)) = '' THEN NULL
        WHEN regexp_matches(
            regexp_replace(trim(CAST(x AS VARCHAR)), '_+$', ''),
            '^[0-9]+(\.[0-9]+)?$'
        )
        THEN printf(
            '%.2f',
            CAST(
                regexp_replace(trim(CAST(x AS VARCHAR)), '_+$', '')
                AS DOUBLE
            )
        )
        ELSE regexp_replace(trim(CAST(x AS VARCHAR)), '_+$', '')
    END
)
''')

con.execute('''
CREATE OR REPLACE VIEW trips_normalized AS
SELECT
    ride_id,
    rideable_type,
    CAST(started_at AS TIMESTAMP) AS started_at,
    CAST(ended_at AS TIMESTAMP) AS ended_at,

    start_station_name,
    canonical_station_id(start_station_id) AS start_station_key,
    start_lat,
    start_lng,

    end_station_name,
    canonical_station_id(end_station_id) AS end_station_key,
    end_lat,
    end_lng,

    member_casual,
    duration_min
FROM trips_with_duration
''')

### Entity-resolution QA

One important case involved `5343.1` and `5343.10`.

Before normalization, the split identifiers created an apparent one-way station-flow problem. After resolving the identifiers to one canonical key, the station showed:

- **7,605 arrivals**
- **7,497 departures**

The apparent extreme imbalance largely disappeared.

This is why entity resolution directly affects business decision quality.

In [ ]:
duplicate_station_ids = con.execute('''
SELECT
    canonical_station_id(start_station_id) AS canonical_key,
    COUNT(DISTINCT CAST(start_station_id AS VARCHAR)) AS raw_id_count,
    list(DISTINCT CAST(start_station_id AS VARCHAR)) AS raw_ids
FROM raw_trips
WHERE start_station_id IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT CAST(start_station_id AS VARCHAR)) > 1
ORDER BY raw_id_count DESC, canonical_key
''').df()

duplicate_station_ids.head(20)

## 7. Build arrivals and departures

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW departures AS
SELECT
    start_station_key AS station_key,
    start_station_name AS station_name,
    CAST(started_at AS DATE) AS service_date,
    EXTRACT('hour' FROM started_at)::INTEGER AS hour,
    start_lat AS latitude,
    start_lng AS longitude,
    COUNT(*) AS departures
FROM trips_normalized
WHERE start_station_key IS NOT NULL
GROUP BY ALL
''')

con.execute('''
CREATE OR REPLACE VIEW arrivals AS
SELECT
    end_station_key AS station_key,
    end_station_name AS station_name,
    CAST(ended_at AS DATE) AS service_date,
    EXTRACT('hour' FROM ended_at)::INTEGER AS hour,
    end_lat AS latitude,
    end_lng AS longitude,
    COUNT(*) AS arrivals
FROM trips_normalized
WHERE end_station_key IS NOT NULL
GROUP BY ALL
''')

## 8. Build the observed station × date × hour table

For each station-hour:

- `net_flow = arrivals - departures`
- `total_activity = arrivals + departures`

Negative net flow indicates bikes leaving faster than they arrive.  
Positive net flow indicates bikes accumulating faster than they leave.

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_day_hour_observed AS
SELECT
    COALESCE(d.station_key, a.station_key) AS station_key,
    COALESCE(d.station_name, a.station_name) AS station_name,
    COALESCE(d.service_date, a.service_date) AS service_date,
    COALESCE(d.hour, a.hour) AS hour,

    COALESCE(d.latitude, a.latitude) AS latitude,
    COALESCE(d.longitude, a.longitude) AS longitude,

    COALESCE(d.departures, 0) AS departures,
    COALESCE(a.arrivals, 0) AS arrivals
FROM departures d
FULL OUTER JOIN arrivals a
    ON d.station_key = a.station_key
   AND d.service_date = a.service_date
   AND d.hour = a.hour
''')

## 9. Create a complete station × active-date × 24-hour grid

This step fixes a denominator problem in persistence metrics.

Without the complete grid, hours with zero rides disappear from the table. A station could then appear to have pressure on `100%` of observed days simply because quiet days were missing from the denominator.

The complete grid explicitly adds zero-activity hours.

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_dates AS
SELECT DISTINCT
    station_key,
    service_date
FROM station_day_hour_observed
''')

con.execute('''
CREATE OR REPLACE VIEW hours AS
SELECT range AS hour
FROM range(24)
''')

con.execute('''
CREATE OR REPLACE VIEW station_day_hour_complete AS
SELECT
    sd.station_key,
    COALESCE(obs.station_name, meta.station_name) AS station_name,
    sd.service_date,
    h.hour,

    COALESCE(obs.latitude, meta.latitude) AS latitude,
    COALESCE(obs.longitude, meta.longitude) AS longitude,

    COALESCE(obs.departures, 0) AS departures,
    COALESCE(obs.arrivals, 0) AS arrivals,

    COALESCE(obs.arrivals, 0) - COALESCE(obs.departures, 0) AS net_flow,
    COALESCE(obs.arrivals, 0) + COALESCE(obs.departures, 0) AS total_activity

FROM station_dates sd
CROSS JOIN hours h

LEFT JOIN station_day_hour_observed obs
    ON sd.station_key = obs.station_key
   AND sd.service_date = obs.service_date
   AND h.hour = obs.hour

LEFT JOIN (
    SELECT
        station_key,
        any_value(station_name) AS station_name,
        median(latitude) AS latitude,
        median(longitude) AS longitude
    FROM station_day_hour_observed
    GROUP BY station_key
) meta
    ON sd.station_key = meta.station_key
''')

## 10. Define daily pressure flags

Imbalance rate:

`net_flow / total_activity`

Project decision thresholds:

- `<= -0.20` → Bike Availability Pressure day
- `>= +0.20` → Dock Availability Pressure day

These are analytical thresholds used for prioritization, not physical inventory thresholds.

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_day_hour_flags AS
SELECT
    *,
    CASE
        WHEN total_activity = 0 THEN 0.0
        ELSE net_flow * 1.0 / total_activity
    END AS imbalance_rate,

    CASE
        WHEN total_activity > 0
         AND (net_flow * 1.0 / total_activity) <= -0.20
        THEN 1 ELSE 0
    END AS bike_pressure_flag,

    CASE
        WHEN total_activity > 0
         AND (net_flow * 1.0 / total_activity) >= 0.20
        THEN 1 ELSE 0
    END AS dock_pressure_flag

FROM station_day_hour_complete
''')

## 11. Aggregate station-hour KPIs

Core metrics:

- Average arrivals
- Average departures
- Average net flow
- Average activity
- Imbalance severity
- Bike-pressure persistence
- Dock-pressure persistence

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_hour_kpis AS
SELECT
    station_key,
    any_value(station_name) AS station_name,
    hour,
    median(latitude) AS latitude,
    median(longitude) AS longitude,

    AVG(arrivals) AS avg_arrivals,
    AVG(departures) AS avg_departures,
    AVG(net_flow) AS avg_net_flow,
    AVG(total_activity) AS avg_total_activity,

    AVG(abs(imbalance_rate)) AS imbalance_severity,

    AVG(bike_pressure_flag) AS bike_pressure_persistence,
    AVG(dock_pressure_flag) AS dock_pressure_persistence,

    COUNT(*) AS active_days

FROM station_day_hour_flags
GROUP BY station_key, hour
''')

## 12. Assign pressure type

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_hour_pressure AS
SELECT
    *,
    CASE
        WHEN avg_net_flow < 0
         AND bike_pressure_persistence >= dock_pressure_persistence
        THEN 'Bike Availability Pressure'

        WHEN avg_net_flow > 0
         AND dock_pressure_persistence > bike_pressure_persistence
        THEN 'Dock Availability Pressure'

        ELSE 'Not Prioritized'
    END AS priority_type,

    GREATEST(
        bike_pressure_persistence,
        dock_pressure_persistence
    ) AS priority_persistence

FROM station_hour_kpis
''')

## 13. Build the composite Priority Score

The project uses percentile normalization so the three components are on comparable 0–1 scales.

Weighting:

- **40% Imbalance Severity**
- **30% Activity**
- **30% Persistence**

The output is scaled to 0–100.

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW station_hour_scored AS
SELECT
    *,

    percent_rank() OVER (
        ORDER BY imbalance_severity
    ) AS severity_score,

    percent_rank() OVER (
        ORDER BY avg_total_activity
    ) AS activity_score,

    percent_rank() OVER (
        ORDER BY priority_persistence
    ) AS persistence_score

FROM station_hour_pressure
''')

con.execute('''
CREATE OR REPLACE VIEW station_hour_priorities AS
SELECT
    station_key,
    station_name,
    hour,
    latitude,
    longitude,

    avg_arrivals,
    avg_departures,
    avg_net_flow,
    avg_total_activity,

    imbalance_severity,
    bike_pressure_persistence,
    dock_pressure_persistence,
    priority_persistence,

    priority_type,

    ROUND(
        100 * (
            0.40 * severity_score
          + 0.30 * activity_score
          + 0.30 * persistence_score
        ),
        1
    ) AS priority_score

FROM station_hour_scored
''')

## 14. Review 8 AM priorities

8 AM is a useful focal hour because the dashboard shows the system-wide morning peak in Bike Availability Pressure.

In [ ]:
top_10_8am = con.execute('''
SELECT
    station_name,
    priority_type,
    priority_score,
    avg_arrivals,
    avg_departures,
    avg_net_flow,
    avg_total_activity,
    imbalance_severity,
    priority_persistence
FROM station_hour_priorities
WHERE hour = 8
  AND priority_type <> 'Not Prioritized'
ORDER BY priority_score DESC
LIMIT 10
''').df()

top_10_8am

## 15. Hourly pressure profile

This query produces the data behind the dashboard's **Rebalancing Pressure by Hour** view.

In [ ]:
hourly_pressure = con.execute('''
SELECT
    hour,
    priority_type,
    COUNT(DISTINCT station_key) AS prioritized_stations
FROM station_hour_priorities
WHERE priority_type <> 'Not Prioritized'
GROUP BY hour, priority_type
ORDER BY hour, priority_type
''').df()

hourly_pressure

## 16. Export the Tableau-ready dataset

In [ ]:
OUTPUT_PATH = "station_hour_priorities.csv"

con.execute(f'''
COPY (
    SELECT *
    FROM station_hour_priorities
    ORDER BY hour, priority_score DESC
)
TO '{OUTPUT_PATH}'
(HEADER, DELIMITER ',')
''')

print(f"Saved: {OUTPUT_PATH}")

## 17. Key analytical takeaways

- **Data quality can create false business signals.** Station-ID fragmentation materially distorted apparent rebalancing pressure until canonical keys were introduced.
- **Outlier detection is not the same as outlier deletion.** Long-duration trips were audited rather than automatically discarded.
- **KPI denominators matter.** Completing the station-hour grid prevented persistence from being overstated.
- **Rebalancing pressure is time-dependent.** Bike-availability pressure is concentrated around the morning commute, while dock-availability pressure becomes more prominent later in the day.
- **Priority Score is a ranking tool, not a stockout probability.**

### Limitations

This analysis does not include real-time bike inventory, dock capacity, weather, special events, truck-routing constraints, or station-to-station travel times. These would be valuable additions for a production rebalancing system.